# 09 — ML Volatility Extension: Does ML Actually Beat GARCH?

**Part 8 of the original project plan** — deliberately built last, and treated as a genuine open question rather than an assumed win for machine learning.

**Research question:** does a Random Forest or XGBoost model, given lagged returns, lagged realised volatility, and rolling distributional features, forecast next-period volatility more accurately than the classical GARCH(1,1) model from `02_volatility_models.ipynb`?

**Why this matters for a risk-team audience:** the honest, defensible answer is often 'no, or only marginally' — GARCH is a well-understood, parsimonious, regulator-accepted model, and beating it meaningfully with ML on a modest sample is genuinely hard. Reporting a close or negative result here is **more credible**, not less, than claiming a large ML win — see the project README's running theme of honest reporting over impressive-sounding claims.

**Method:**
1. Build features: lagged returns, lagged realised volatility, rolling skew/kurtosis
2. Chronological (non-shuffled) train/test split — critical for time series
3. Fit Random Forest and XGBoost regressors
4. Compare against the GARCH(1,1) forecast from notebook 02, using the *same* RMSE / MAE / QLIKE metrics for a fair, apples-to-apples comparison
5. Inspect feature importances — does the model learn something sensible (e.g. lagged realised volatility should matter most), or does it look like it's fitting noise?

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.volatility import fit_garch
from src.ml_volatility import (
    build_features, build_target, train_test_split_time_series,
    fit_random_forest, fit_xgboost, compare_ml_vs_garch, feature_importance_report
)

port_returns = pd.read_csv('../data/portfolio_returns.csv', index_col=0, parse_dates=True).iloc[:, 0]


## 1. Fit GARCH (the baseline to beat)

Reuses the same GARCH(1,1) fit from notebook 02 — its in-sample conditional volatility series acts as the classical baseline forecast throughout this notebook.

In [ ]:
garch_fit = fit_garch(port_returns)
garch_conditional_vol = pd.Series(
    garch_fit.conditional_volatility / 100,  # unscale from the *100 used inside fit_garch()
    index=port_returns.index,
)
garch_conditional_vol.tail()


## 2. Run the full comparison

`compare_ml_vs_garch()` builds features/target, does a chronological train/test split, fits both ML models, and evaluates all three approaches on identical out-of-sample data using the same metrics as notebook 02.

In [ ]:
comparison = compare_ml_vs_garch(port_returns, garch_conditional_vol, test_size=0.25)
comparison.round(6)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ['RMSE', 'MAE', 'QLIKE']):
    comparison[metric].plot(kind='bar', ax=ax, color=['#2E4A6B', '#5B8C5A', '#B0413E'])
    ax.set_title(metric)
    ax.set_ylabel('' )
plt.tight_layout()
plt.savefig('../results/figures/ml_vs_garch_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Feature importance — sanity check

If lagged realised volatility doesn't dominate here, that's worth investigating before trusting the model — it would suggest the model is picking up spurious patterns rather than the genuine volatility-clustering signal GARCH is built specifically to capture.

In [ ]:
X = build_features(port_returns)
y = build_target(port_returns)
X_train, X_test, y_train, y_test = train_test_split_time_series(X, y)

rf_model = fit_random_forest(X_train, y_train)
importances = feature_importance_report(rf_model, X_train.columns.tolist())

fig, ax = plt.subplots(figsize=(7, 5))
importances.head(10).plot(kind='barh', ax=ax)
ax.invert_yaxis()
ax.set_title('Random Forest — Top 10 Feature Importances')
plt.tight_layout()
plt.savefig('../results/figures/ml_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Honest conclusion

*(Fill in once run on real data: did either ML model beat GARCH on QLIKE — the metric that penalises under-prediction of volatility most heavily, and so matters most for risk management? By how much, and is that gap large enough to justify the extra complexity, reduced interpretability, and lack of regulatory precedent versus a well-understood GARCH model? If GARCH wins or the gap is small, say so plainly — that is a genuinely useful, defensible finding for a risk-team audience, consistent with this project's approach throughout.)*

---

**This completes the full project**, from Part 1's general market-risk pipeline through Part 2's dissertation-derived Climate/Hybrid VaR, Part 3's earnings-event NLP fusion, to this final honest test of whether machine learning actually adds value over classical volatility models — the appropriate way to end a risk-modelling project: not with an assumed win, but with a tested answer.